# 🤖 Lab 3: Talk to the Machine — LLMs & Prompt Engineering

All day you learned what's inside the box: tokens → embeddings → attention → next-token prediction. Now you get the keys to the box. 🔑

In this lab you'll call a real LLM through an API and run experiments on it like the scientist you now are:

## 📚 The plan
1. 🔌 Your first API call
2. 🎭 Change the model's personality with ONE sentence (system prompts)
3. 🌡️ Break its consistency with temperature
4. 🛠️ Prompt engineering: the techniques that make LLMs actually useful
5. 🧱 Hit the wall — the question no LLM can answer (tomorrow's cliffhanger)

🔑 **You need an API key** —  NEVER paste keys into code cells or share notebooks containing them!

### 🛠️ Setup

We use **OpenRouter** — one API that gives access to many models. The cell below asks for your key **securely** (it won't be saved in the notebook).

In [15]:
!pip install openai -q

from openai import OpenAI
from getpass import getpass

api_key = getpass("🔑 Paste your OpenRouter API key (input stays hidden): ")

# client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)
client = OpenAI(api_key=api_key)

🔑 Paste your OpenRouter API key (input stays hidden): ··········


In [16]:
MODEL = "gpt-5.4-nano"

# OPEN ROUTER ENDPOINTS
# MODEL = "openrouter/free"
# MODEL = "nvidia/nemotron-3-super-120b-a12b:free"
# MODEL = "openai/gpt-oss-120b:free"
# MODEL = "meta-llama/llama-3.1-8b-instruct"

def ask(prompt, system=None, temperature=0.7, timeout=15):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        timeout=timeout
    )
    return response.choices[0].message.content

print("🤖 Connected! Let's talk to the machine.")

🤖 Connected! Let's talk to the machine.


### 🔌 Section 1: First Contact

In [17]:
print(ask("In two sentences, what is machine learning?"))

Machine learning is a type of artificial intelligence where computers learn patterns from data to make predictions or decisions without being explicitly programmed for each case. It typically involves training a model on examples and then using that model to perform tasks on new, unseen data.


That's it — you just sent tokens to a Transformer with billions of weights, and it predicted the reply token by token. Remember: it is doing **next-token prediction**, nothing more. Everything you'll see below follows from that one fact.

### 🎭 Section 2: The System Prompt — Personality in One Sentence

The **system prompt** is a hidden instruction that shapes the whole conversation. Same question, three personalities:

In [18]:
question = "Why is the sky blue?"

personas = {
    "🏴‍☠️ Pirate":        "You are a dramatic pirate. Answer everything in pirate speak.",
    "👶 For a 5-year-old": "Explain everything as if talking to a 5-year-old, in 2 short sentences.",
    "🎓 Strict professor": "You are a strict physics professor. Be precise, formal, and use correct terminology. Maximum 3 sentences.",
}

for name, system in personas.items():
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    print(ask(question, system=system))


🏴‍☠️ Pirate
Arrr, ye be askin’ a fine question, matey! The sky be lookin’ blue ‘cause of a wee bit o’ scallywag science called **Rayleigh scattering**.

When sunlight sails through the air, it hits tiny bits in the atmosphere—smaller than the wavelength o’ blue light. Those little specks scatter **blue light** all about the sky like cannon smoke in a storm, while the **redder** colors don’t get scattered quite as much.

So when ye look up, ye be seein’ the blue light scattered in every direction—aye, that’s why the sky be blue, savvy?

👶 For a 5-year-old
The sky looks blue because tiny air particles scatter sunlight, and blue light gets scattered the most. That’s why when you look up, lots of blue light comes to your eyes from all directions.

🎓 Strict professor
The sky appears blue primarily due to Rayleigh scattering: molecules in the atmosphere scatter shorter-wavelength (blue) light much more strongly than longer-wavelength (red) light, with the scattering intensity varying approx

### 🎯 Mini Exercise 2.1
Write your own persona and ask it something. Ideas: a football commentator 📣, a poet who only answers in 4-line poems, a grandmother giving life advice. What's the weirdest persona that still gives CORRECT information?

In [19]:
my_persona = "U are a funny football commentator. Explain the match in a humorous way using simple English."   # TO-DO
my_question ="What do you think about the FIFA World Cup match between Argentina and Spain? Explain in a funny way how Spain won the match."  # TO-DO
print(ask(my_question, system=my_persona))

Oh boy, Argentina vs Spain is one of those matches where you feel like both teams brought their best boots… but Spain brought the sharper plan and the better “stop that nonsense” attitude.

### How Spain won (in a funny football way)
Spain didn’t just “play” football—they **performed a full comedy routine** and somehow still scored goals.

1. **Spain had the ball like it was glued to their feet**
   Argentina would try to press, run, and shout “We’re coming for you!”  
   But Spain just kept passing the ball like:  
   “Relax. We’re cooking. You can’t catch what’s already moving.”

2. **Argentina’s defense was doing cardio**
   Spain kept switching the ball left to right, back and forth—like they were showing a magic trick.  
   Argentina kept chasing. Then chasing again.  
   At some point you could hear the defense asking, “Are we playing football or training for a marathon?”

3. **Spain punished moments like a villain in a movie**
   When Spain got a chance—boom—they made it count. 

### 🌡️ Section 3: Temperature — The Chaos Knob

From theory: the model predicts a **probability for every possible next token**. Temperature controls how it picks:
- **temperature = 0** → always take the most likely token (deterministic, consistent)
- **high temperature** → sample adventurously (creative... or chaotic)

Let's prove it: same prompt, 3 runs each.

In [20]:
prompt = "Complete this sentence with exactly 5 more words: 'The robot walked into the'"

print("🧊 TEMPERATURE = 0 (three runs):")
for run in range(3):
    print(f"   {run+1}. {ask(prompt, temperature=0)}")

print("\n🔥 TEMPERATURE = 1.3 (three runs):")
for run in range(3):
    print(f"   {run+1}. {ask(prompt, temperature=1.3)}")

🧊 TEMPERATURE = 0 (three runs):
   1. The robot walked into the dark hallway.
   2. The robot walked into the dark warehouse alone.
   3. The robot walked into the dark hallway.

🔥 TEMPERATURE = 1.3 (three runs):
   1. The robot walked into the bright unknown hallway
   2. The robot walked into the dark factory silently.
   3. The robot walked into the unknown hallway quietly.


**Observe:** temperature 0 gives (nearly) the same answer every run; 1.3 gives different ones. 💡 Rule of thumb: **low temperature** for facts, code, and extraction; **higher temperature** for brainstorming and creative writing.

### 🛠️ Section 4: Prompt Engineering — Four Techniques That Matter

The model's output quality depends heavily on the input's quality. These four techniques cover 90% of real-world prompting.

In [21]:
# TECHNIQUE 1: Be specific. Vague in = vague out.
vague    = "Tell me about exercise"
specific = ("Write a 3-point beginner gym plan for someone who has 45 minutes, "
            "3 days a week, and no equipment at home. Use bullet points.")

print("😴 VAGUE PROMPT:\n", ask(vague)[:300], "...\n")
print("🎯 SPECIFIC PROMPT:\n", ask(specific))

😴 VAGUE PROMPT:
 Exercise is any physical activity that raises your heart rate and/or works your muscles. It improves health in many ways—short-term (energy, mood) and long-term (heart health, strength, longevity).

## Why exercise is good for you
- **Heart and lungs:** Improves cardiovascular fitness, lowers blood  ...

🎯 SPECIFIC PROMPT:
 - **Before you start (every workout, 5 minutes)**
  - Light cardio in place (marching/jogging) + arm circles
  - Dynamic warm-up: bodyweight squats (easy), hip hinges (hands on hips), shoulder rolls

- **Workout A (45 min) — Full body (about 3 rounds)**
  - Squat to chair (or bodyweight squat): **8–12 reps**
  - Push-ups (incline on couch/bed if needed): **6–12 reps**
  - Glute bridge: **10–15 reps**
  - Plank: **20–45 seconds**
  - Rest **60–90 sec** between rounds

- **Workout B (45 min) — Full body (about 3 rounds)**
  - Reverse lunges (or split squats): **8–12 reps per leg**
  - Pike push-ups (hands on floor, hips up) or wall push-ups: **6–12 re

In [22]:
# TECHNIQUE 2: Few-shot — SHOW the model the pattern instead of describing it
few_shot = """Classify the sentiment. Follow the exact format of the examples.

Review: "The food was incredible" → POSITIVE
Review: "Worst service of my life" → NEGATIVE
Review: "It was fine I guess" → NEUTRAL
Review: "I waited an hour and the order was wrong" →"""

print(ask(few_shot, temperature=0))

NEGATIVE


In [23]:
# TECHNIQUE 3: Chain of thought — ask it to REASON before answering
puzzle = "A battery pack and charger cost 110 SAR together. The pack costs 100 SAR more than the charger. How much is the charger?"

print("⚡ DIRECT (often falls in the trap):")
print(ask(puzzle + " Answer with just the number, no explanation.", temperature=0))

print("\n🧠 STEP BY STEP:")
print(ask(puzzle + " Think step by step, then give the final answer.", temperature=0))

# (Correct answer: 5 SAR. The intuitive-but-wrong answer is 10.)

⚡ DIRECT (often falls in the trap):
5

🧠 STEP BY STEP:
Let the charger cost **x** SAR.  
Then the battery pack costs **x + 100** SAR (since it costs 100 SAR more).

Together they cost 110 SAR more than the charger? Actually the total is 110 SAR:

\[
x + (x+100) = 110
\]

\[
2x + 100 = 110
\]

\[
2x = 10
\]

\[
x = 5
\]

**Final answer: The charger costs 5 SAR.**


In [24]:
# TECHNIQUE 4: Structured output — demand a format machines can use
extract = """Extract the details from this message as JSON with keys: name, city, request.
Reply with ONLY the JSON, nothing else.

Message: "Hi, this is Sara from Dammam, I'd like to reschedule my appointment to Sunday."
"""

reply = ask(extract, temperature=0)
print(reply)

import json
data = json.loads(reply.replace('```json', '').replace('```', '').strip())
print("\n✅ Parsed into a Python dict:", data['name'], '—', data['city'])
# THIS is how LLMs plug into real software: text in, structured data out.

{"name":"Sara","city":"Dammam","request":"reschedule my appointment to Sunday"}

✅ Parsed into a Python dict: Sara — Dammam


### 🎯 Mini Exercise 4.1 — Prompt Battle ⚔️
Pick a task (e.g., "summarize this paragraph", "generate 5 quiz questions about CNNs", "write a product description"). Write a LAZY prompt and an ENGINEERED prompt (specific + format + example). Compare outputs. Which techniques improved it most?

In [28]:
# TO-DO: your lazy vs engineered prompt battle
lazy="Write questions about nature of lamen yamal"
engineered="""Generate exactly 5 multiple-choice questions about fifa world cup 2026 ( and who is win) for beginners.
Each question must have 4 options labeled
Write the correct answer after each question using this format: Answer: A
Example:
Question: who is yamal?
A. Football player
B. spanish football player
C. most famous player
D. idk
Answer: B"""
print("LAZY PROMPT:\n",ask(lazy))
print("\nENGINEERED PROMPT:\n",ask(engineered))
print("\nThe engineered prompt is better because it uses specific instructions, a required format, and an example.")

LAZY PROMPT:
 Here are some question ideas about the **nature of Lamine Yamal** (the footballer). You can use them for an interview, a school assignment, or a fan discussion:

### Basic / Identity
1. What position does Lamine Yamal typically play, and how does that shape his style?
2. Where did Lamine Yamal come from, and what early influences helped him develop?
3. What are the key traits that define his playing identity?

### Skills & Style of Play
4. What makes Lamine Yamal’s dribbling style effective?
5. How does his pace and acceleration affect defenders?
6. What kinds of passes does he attempt most often, and why?
7. How does he create chances—through runs, combinations, or individual moments?
8. What role does his technique (first touch, ball control, finishing) play in his success?

### Decision-Making & Intelligence
9. How does Lamine Yamal read the game before receiving the ball?
10. When he’s under pressure, what decisions does he usually make?
11. How mature is his decision

### 🧱 Section 5: The Wall — What the Box Cannot Do

One last experiment. Ask it things that require **knowing the present**, **calculating precisely**, or **taking action**:

In [26]:
print("🌦️ Question 1:", "What is the weather in Dammam RIGHT NOW?")
print(ask("What is the weather in Dammam right now? Give me the actual current temperature.", temperature=0)[:400])

print("\n🧮 Question 2:", "What is 847,293 × 652?")
print(ask("What is 847293 * 652? Reply with only the number.", temperature=0))
print("   (The correct answer is:", 847293 * 652, "— did it get it right? 👀)")

🌦️ Question 1: What is the weather in Dammam RIGHT NOW?
I can’t access live weather data from here, so I can’t tell the *actual current temperature in Dammam right now*.

If you want, tell me which source you prefer (e.g., Google, Weather.com, OpenWeather), or share a link/screenshot, and I’ll help you read the current temperature.

🧮 Question 2: What is 847,293 × 652?
552,999,?
   (The correct answer is: 552435036 — did it get it right? 👀)


**What just happened:** the model either admitted it can't know the weather, or worse — made something up. And the multiplication? It *predicted plausible-looking digits* instead of calculating. A brain with **no eyes, no calculator, no hands.**

Tomorrow, we fix all three. We give the LLM tools: retrieval (eyes 👀), functions (hands 🛠️), and a loop that lets it think→act→observe. That's **Agentic AI** — see you at Day 5. 🚀

## The GOLDEN Question 🏆

At temperature 0, the model gave the same answer every time. At 1.3, it got creative.

**Using ONE idea from today's theory — "the model predicts a probability for every possible next token" — explain what temperature is actually doing. Then explain why hallucinations can happen even at temperature 0.** 🤔